# 02 - Tabular Models for Dementia Prediction

This notebook demonstrates training and evaluating tabular models for dementia classification using clinical and demographic data.

---

## Outline
- Data Loading and Preparation
- Train-Test Split
- Model Training (Logistic Regression, Random Forest, Gradient Boosting)
- Model Evaluation
- Performance Comparison
- Model Serialization

---

In [ ]:
# Import required libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import pickle

# Add src to path
sys.path.append('../src')
from data_loading import load_clinical_data
from preprocessing import get_fit_transform
from tabular_models import train_logistic_regression, train_random_forest, train_gbm

# Display settings
pd.set_option('display.max_columns', 100)
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Data Loading and Preparation

Load preprocessed clinical data and prepare features and target.

In [ ]:
# Load clinical data
# NOTE: Update path to your actual data file
clinical_path = '../data/raw/clinical.csv'

try:
    df = load_clinical_data(clinical_path)
    print(f"Loaded data with shape: {df.shape}")
    display(df.head())
except FileNotFoundError:
    print("Data file not found. Please ensure data is downloaded and placed in the correct location.")
    print("See data/README_data.md for instructions.")
    df = None

In [ ]:
# Define features and target
# NOTE: Adjust these based on your actual dataset columns
if df is not None:
    # Example feature columns (adjust to match your data)
    numeric_features = ['Age', 'EDUC', 'MMSE', 'eTIV', 'nWBV', 'ASF']
    categorical_features = ['M/F']
    target_column = 'CDR'  # Clinical Dementia Rating or similar target
    
    # Remove rows with missing target
    df_clean = df.dropna(subset=[target_column])
    
    # Separate features and target
    X = df_clean[[col for col in numeric_features + categorical_features if col in df_clean.columns]]
    y = df_clean[target_column]
    
    # For binary classification, convert CDR to binary (0 vs >0)
    y_binary = (y > 0).astype(int)
    
    print(f"Features shape: {X.shape}")
    print(f"Target distribution:\n{y_binary.value_counts()}")

## 2. Preprocessing and Train-Test Split

In [ ]:
if df is not None:
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
    )
    
    # Preprocess data
    actual_numeric = [f for f in numeric_features if f in X.columns]
    actual_categorical = [f for f in categorical_features if f in X.columns]
    
    X_train_processed, preprocessor = get_fit_transform(
        X_train, actual_numeric, actual_categorical
    )
    X_test_processed = preprocessor.transform(X_test)
    
    # Convert to DataFrame for consistency
    feature_names = preprocessor.get_feature_names_out()
    X_test_processed = pd.DataFrame(
        X_test_processed, columns=feature_names, index=X_test.index
    )
    
    print(f"Training set: {X_train_processed.shape}")
    print(f"Test set: {X_test_processed.shape}")

## 3. Model Training

Train multiple tabular models for comparison.

In [ ]:
# Dictionary to store models
models = {}

if df is not None:
    # Train Logistic Regression
    print("Training Logistic Regression...")
    models['Logistic Regression'] = train_logistic_regression(
        X_train_processed, y_train, max_iter=1000, random_state=42
    )
    
    # Train Random Forest
    print("Training Random Forest...")
    models['Random Forest'] = train_random_forest(
        X_train_processed, y_train, n_estimators=100, random_state=42, n_jobs=-1
    )
    
    # Train Gradient Boosting Machine
    print("Training Gradient Boosting...")
    models['Gradient Boosting'] = train_gbm(
        X_train_processed, y_train, n_estimators=100, random_state=42
    )
    
    print("All models trained successfully!")

## 4. Model Evaluation

In [ ]:
# Evaluate each model
results = {}

if df is not None:
    for name, model in models.items():
        # Predictions
        y_pred = model.predict(X_test_processed)
        y_pred_proba = model.predict_proba(X_test_processed)[:, 1]
        
        # Metrics
        auc = roc_auc_score(y_test, y_pred_proba)
        
        results[name] = {
            'predictions': y_pred,
            'probabilities': y_pred_proba,
            'auc': auc
        }
        
        print(f"\n{'='*60}")
        print(f"Model: {name}")
        print(f"{'='*60}")
        print(f"AUC-ROC: {auc:.4f}")
        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))
        
        # Confusion Matrix
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title(f'Confusion Matrix - {name}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.show()

## 5. Performance Comparison

In [ ]:
# Compare ROC curves
if df is not None:
    plt.figure(figsize=(10, 8))
    
    for name, result in results.items():
        fpr, tpr, _ = roc_curve(y_test, result['probabilities'])
        plt.plot(fpr, tpr, label=f"{name} (AUC={result['auc']:.3f})")
    
    plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves Comparison')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    # Performance summary table
    summary_df = pd.DataFrame({
        'Model': list(results.keys()),
        'AUC-ROC': [r['auc'] for r in results.values()]
    }).sort_values('AUC-ROC', ascending=False)
    
    print("\nPerformance Summary:")
    display(summary_df)

## 6. Model Serialization

Save trained models for later use.

In [ ]:
# Save models
if df is not None:
    os.makedirs('../models', exist_ok=True)
    
    for name, model in models.items():
        model_filename = f"../models/{name.lower().replace(' ', '_')}.pkl"
        with open(model_filename, 'wb') as f:
            pickle.dump(model, f)
        print(f"Saved {name} to {model_filename}")
    
    # Save preprocessor
    with open('../models/preprocessor.pkl', 'wb') as f:
        pickle.dump(preprocessor, f)
    print("Saved preprocessor to ../models/preprocessor.pkl")

## Summary

This notebook demonstrated:
- Loading and preprocessing clinical data
- Training multiple tabular models (Logistic Regression, Random Forest, Gradient Boosting)
- Evaluating model performance with various metrics
- Comparing models using ROC curves
- Saving trained models for future use

### Next Steps
- Proceed to notebook 03 for CNN models on MRI imaging data
- Use saved models in ensemble fusion (notebook 04)
- Apply explainability techniques (notebook 05)